# Randomized Benchmarking Calibration — CGCS / Triplet-Phase-Lock Starter Notebook

**Repository:** `quantum-hardware-company`  
**Directory:** `calibration/rb/`

This notebook initializes a randomized benchmarking (RB) calibration workflow:

1. Generate synthetic RB sequence survival data.
2. Fit an exponential RB decay model.
3. Estimate decay parameter and error-per-gate proxy.
4. Analyze residual structure.
5. Compute a CGCS / Triplet-Phase-Lock metric using cosine similarity.
6. Save figures and a machine-readable JSON summary.

## Principle

RB converts gate behavior into measurable decay.

Rabi calibrates amplitude.  
Ramsey calibrates phase/coherence.  
RB checks whether those controls survive randomized gate composition.


## RB model

We use the standard RB decay primitive:

$$
P(m)=A p^m + B
$$

where:

- $m$ is Clifford / sequence length,
- $A$ is SPAM-sensitive amplitude,
- $p$ is the RB decay parameter,
- $B$ is the asymptotic survival floor.

For a simple one-qubit proxy, we estimate:

$$
r \approx \frac{1-p}{2}
$$

This starter notebook uses synthetic data and is meant to establish the calibration workflow before replacing data with experiment outputs.


In [ ]:
import os
import json

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

np.random.seed(9423)

FIG_DIR = "figures"
RESULTS_DIR = "results"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)


In [ ]:
def rb_model(m, A, p, B):
    """Randomized benchmarking survival model."""
    return A * (p ** m) + B


def fit_model(model, x, y, p0, bounds=(-np.inf, np.inf), sigma=None):
    """Fit model to data and return best-fit params + covariance."""
    params, cov = curve_fit(
        model,
        x,
        y,
        p0=p0,
        bounds=bounds,
        sigma=sigma,
        absolute_sigma=sigma is not None,
        maxfev=30000,
    )
    return params, cov


def residuals(model, x, y, params):
    """Observed minus fitted values."""
    return y - model(x, *params)


def phase_lock_metric(signal, fit):
    """Cosine similarity between observed signal and fitted model."""
    dot = np.dot(signal, fit)
    norm = np.linalg.norm(signal) * np.linalg.norm(fit)
    if norm == 0:
        return np.nan
    return dot / norm


def is_phase_locked(metric, threshold=1 / np.sqrt(2)):
    """CGCS threshold: cos(theta) >= 1/sqrt(2), equivalent to <= 45 degrees."""
    return metric >= threshold


def autocorrelation(x):
    """Simple non-normalized autocorrelation for residual structure checks."""
    x = x - np.mean(x)
    result = np.correlate(x, x, mode="full")
    return result[result.size // 2:]


## Generate synthetic RB calibration data

This cell stands in for experimental RB readout data.

Synthetic sequence lengths are repeated several times to mimic repeated random Clifford sequences per length.


In [ ]:
# RB sequence lengths
m_values = np.array([1, 2, 4, 8, 12, 16, 24, 32, 48, 64, 96, 128, 192])

# Ground-truth synthetic parameters
true_A = 0.46
true_p = 0.987
true_B = 0.50
true_params = [true_A, true_p, true_B]

# Simulate repeated random sequences per length
n_repeats = 30

m_obs = np.repeat(m_values, n_repeats)
y_clean = rb_model(m_obs, *true_params)

# Heteroscedastic measurement noise + small structured length-dependent deviation
measurement_noise = 0.018 * np.random.randn(len(m_obs))
structured_deviation = 0.006 * np.sin(0.11 * m_obs)

y_obs = y_clean + measurement_noise + structured_deviation
y_obs = np.clip(y_obs, 0, 1)

# Aggregate by length: mean and standard error
means = []
sems = []
for m in m_values:
    samples = y_obs[m_obs == m]
    means.append(np.mean(samples))
    sems.append(np.std(samples, ddof=1) / np.sqrt(len(samples)))

means = np.array(means)
sems = np.array(sems)

print("True parameters:")
print(f"A = {true_A:.6f}")
print(f"p = {true_p:.6f}")
print(f"B = {true_B:.6f}")
print(f"one-qubit error proxy r ≈ {(1 - true_p) / 2:.6f}")


## Fit RB decay model


In [ ]:
initial_guess = [0.45, 0.98, 0.50]

# Bounds keep fit physically interpretable:
# A in [0, 1], p in [0, 1], B in [0, 1]
lower_bounds = [0.0, 0.0, 0.0]
upper_bounds = [1.0, 1.0, 1.0]

params, cov = fit_model(
    rb_model,
    m_values,
    means,
    p0=initial_guess,
    bounds=(lower_bounds, upper_bounds),
    sigma=sems,
)

A_fit, p_fit, B_fit = params
param_std = np.sqrt(np.diag(cov))
y_fit = rb_model(m_values, *params)

error_per_gate_proxy = (1 - p_fit) / 2
error_per_gate_proxy_std = param_std[1] / 2

print("Fit parameters:")
print(f"A = {A_fit:.6f} ± {param_std[0]:.6f}")
print(f"p = {p_fit:.6f} ± {param_std[1]:.6f}")
print(f"B = {B_fit:.6f} ± {param_std[2]:.6f}")
print(f"one-qubit error proxy r ≈ {error_per_gate_proxy:.6f} ± {error_per_gate_proxy_std:.6f}")


## Plot RB decay fit


In [ ]:
m_dense = np.linspace(m_values.min(), m_values.max(), 500)
y_dense = rb_model(m_dense, *params)

plt.figure(figsize=(9, 5))
plt.errorbar(
    m_values,
    means,
    yerr=sems,
    fmt="o",
    capsize=3,
    label="mean survival data",
)
plt.plot(m_dense, y_dense, linewidth=2, label="RB exponential fit")
plt.xlabel("sequence length m")
plt.ylabel("survival probability")
plt.title("Randomized benchmarking: measured decay and fitted model")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/rb_01_decay_fit.png", dpi=300)
plt.savefig(f"{FIG_DIR}/rb_01_decay_fit.pdf")

plt.show()


## Linearized decay view

This diagnostic plots the fitted decay after subtracting the fitted floor $B$.

It is useful as a sanity check, but the primary fit remains the nonlinear RB model above.


In [ ]:
positive = means > B_fit
m_linear = m_values[positive]
log_decay = np.log(np.maximum(means[positive] - B_fit, 1e-12))
log_fit = np.log(np.maximum(rb_model(m_linear, *params) - B_fit, 1e-12))

plt.figure(figsize=(8, 5))
plt.scatter(m_linear, log_decay, label="log(mean - fitted floor)")
plt.plot(m_linear, log_fit, linewidth=2, label="linearized fitted decay")
plt.xlabel("sequence length m")
plt.ylabel("log survival above floor")
plt.title("RB linearized decay diagnostic")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/rb_02_linearized_decay.png", dpi=300)
plt.savefig(f"{FIG_DIR}/rb_02_linearized_decay.pdf")

plt.show()


## Residual analysis

Residuals are treated as measurable structure unless demonstrated random.


In [ ]:
res = residuals(rb_model, m_values, means, params)
weighted_res = res / sems

rmse = np.sqrt(np.mean(res**2))
weighted_rmse = np.sqrt(np.mean(weighted_res**2))
mean_res = np.mean(res)
std_res = np.std(res)

print(f"Residual mean   = {mean_res:.8f}")
print(f"Residual std    = {std_res:.8f}")
print(f"Fit RMSE        = {rmse:.8f}")
print(f"Weighted RMSE   = {weighted_rmse:.8f}")


In [ ]:
plt.figure(figsize=(9, 4))
plt.axhline(0, linewidth=1)
plt.errorbar(m_values, res, yerr=sems, fmt="o-", capsize=3)
plt.xlabel("sequence length m")
plt.ylabel("residual: mean observed - fit")
plt.title("RB calibration residuals")
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/rb_03_residuals.png", dpi=300)
plt.savefig(f"{FIG_DIR}/rb_03_residuals.pdf")

plt.show()


## Weighted residuals

Weighted residuals highlight whether deviations are large relative to measurement uncertainty.


In [ ]:
plt.figure(figsize=(9, 4))
plt.axhline(0, linewidth=1)
plt.plot(m_values, weighted_res, marker="o", linewidth=1)
plt.xlabel("sequence length m")
plt.ylabel("weighted residual")
plt.title("RB weighted residuals")
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/rb_04_weighted_residuals.png", dpi=300)
plt.savefig(f"{FIG_DIR}/rb_04_weighted_residuals.pdf")

plt.show()


## Residual autocorrelation

This diagnostic checks whether residuals contain structure over sequence length rather than behaving like independent random noise.


In [ ]:
ac = autocorrelation(res)

plt.figure(figsize=(7, 4))
plt.plot(ac[: len(ac)])
plt.title("RB residual autocorrelation (structure check)")
plt.xlabel("lag")
plt.ylabel("correlation")
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/rb_05_residual_autocorr.png", dpi=300)
plt.savefig(f"{FIG_DIR}/rb_05_residual_autocorr.pdf")

plt.show()


## CGCS / Triplet-Phase-Lock metric

We compute cosine similarity between aggregated measured survival data and fitted RB model.

The working CGCS threshold is:

$$
\cos\theta \geq \frac{1}{\sqrt{1^2+1^2}}
$$

This is the 45° constraint gate.


In [ ]:
metric = phase_lock_metric(means, y_fit)
threshold = 1 / np.sqrt(2)
locked = is_phase_locked(metric, threshold=threshold)
angle_deg = np.degrees(np.arccos(np.clip(metric, -1, 1)))

print(f"CGCS phase-lock metric = {metric:.6f}")
print(f"CGCS threshold         = {threshold:.6f}")
print(f"Angle                  = {angle_deg:.3f} degrees")
print(f"Phase locked?          = {locked}")


## CGCS interpretation

A phase-lock angle far below 45° indicates that measured RB decay remains inside the constraint region.

For calibration, this means the fitted decay explains aggregate gate-survival behavior before downstream noise-mitigation or control-stack adjustments.


In [ ]:
plt.figure(figsize=(7, 4))
labels = ["fit alignment", "45° threshold"]
values = [metric, threshold]

bars = plt.bar(labels, values)
plt.axhline(threshold, linewidth=1, linestyle="--")
plt.ylim(0, 1.05)
plt.ylabel("cosine similarity")
plt.title("CGCS phase-lock check")

for bar, value in zip(bars, values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.015,
        f"{value:.3f}",
        ha="center",
        va="bottom",
    )

plt.tight_layout()

plt.savefig(f"{FIG_DIR}/rb_06_phase_lock.png", dpi=300)
plt.savefig(f"{FIG_DIR}/rb_06_phase_lock.pdf")

plt.show()


## Calibration summary


In [ ]:
summary = {
    "A_fit": float(A_fit),
    "p_fit": float(p_fit),
    "B_fit": float(B_fit),
    "A_fit_std": float(param_std[0]),
    "p_fit_std": float(param_std[1]),
    "B_fit_std": float(param_std[2]),
    "one_qubit_error_proxy": float(error_per_gate_proxy),
    "one_qubit_error_proxy_std": float(error_per_gate_proxy_std),
    "residual_rmse": float(rmse),
    "weighted_rmse": float(weighted_rmse),
    "phase_lock_metric": float(metric),
    "phase_lock_threshold": float(threshold),
    "phase_lock_angle_degrees": float(angle_deg),
    "phase_locked": bool(locked),
}

summary


## Save calibration summary


In [ ]:
with open(f"{RESULTS_DIR}/rb_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved {RESULTS_DIR}/rb_summary.json")


## Zip outputs for download from Colab

Run this cell after all figures/results have been generated.


In [ ]:
!zip -r rb_outputs.zip figures results


## Next steps

This notebook completes the first calibration trio:

- `calibration/rabi/` → amplitude / drive response
- `calibration/ramsey/` → phase / detuning / dephasing
- `calibration/rb/` → randomized gate-survival decay

From here, the repository can move into:

- `calibration/drift/` for stability over time,
- `noise-mitigation/` for structured residual modeling,
- `control-stack/` for closed-loop pulse tuning,
- `control-electronics/` for timing and filtering constraints.

Guiding rule:

> Start where physics shows up.
